# CARE — IAM Hardening Demo

Full CARE pipeline on a sample AWS IAM state:
encode → risk → curvature → ridge analysis → recommendations → hardening simulation → visualisation.

**Theory:** Unified Attractor Grammar (Byte 2026), DOI: 10.5281/zenodo.19394700

## 0. Setup

In [ ]:
import sys, pathlib
for p in ['../..', '.']:
    if p not in sys.path: sys.path.insert(0, p)
import json, numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'font.size': 9})
%matplotlib inline
print("Ready.")

In [ ]:
from care.models.state_encoder import encode_state
from care.models.risk_potential import get_potential, PrivilegeRisk, BlastRadiusRisk
from care.models.curvature import curvature_info
from care.models.ridge import analyse_ridge
from care.models.recommend import recommend_actions
from care.adapters.constitutional_os import propose_delta
from care.viz.plots import eigenvalue_spectrum, before_after, curvature_basin_2d, risk_over_time
print("CARE modules imported.")

## 1. IAM State

In [ ]:
with open(pathlib.Path(__file__).parent / 'iam_state.json' if '__file__' in dir()
         else 'iam_state.json') as f:
    iam_state = json.load(f)
for k, v in iam_state.items():
    print(f"  {k:35s}: {v}")

In [ ]:
x = encode_state(iam_state, target_dim=16)
print(f"x shape: {x.shape}   x[:6]: {x[:6].round(3)}")

## 2. Risk & Curvature

R(x) = Σ wᵢ xᵢ² with w₀..₃ = 5 (privilege dims), w₄.. = 1.

**UAG Theorem 3:** eigenvalues of H(x) = stability + stiffness + flow direction.

In [ ]:
potential = PrivilegeRisk(n_privileged=4, privilege_weight=5.0)
result = curvature_info(x, potential, backend='numpy')
print(f"Risk R(x)        = {result.risk:.2f}")
print(f"Gradient norm    = {np.linalg.norm(result.gradient):.4f}")
print(f"Eigenvalues[:8]  = {result.eigenvalues[:8].round(4)}")

In [ ]:
fig = eigenvalue_spectrum(result.eigenvalues[:8],
                          title="Eigenvalue spectrum — before hardening")
plt.tight_layout(); plt.show()

## 3. Ridge Analysis

**UAG Theorem 4:** min |λ_min| on ∂B → minimum-action escape route.

Kramers proxy ≈ exp(−2ΔR / |λ_min|): higher = easier escape from safe basin.

In [ ]:
ridge = analyse_ridge(result)
print(f"Severity:            {ridge.severity.upper()}")
print(f"Softest λ:           {ridge.softest_eigenvalue:.4f}  (dim {ridge.softest_index})")
print(f"Kramers proxy:       {ridge.kramers_proxy:.4f}")
print(f"Negative eigenvals:  {ridge.negative_eigenvalues}")

## 4. Hardening Recommendations

In [ ]:
actions = recommend_actions(result, ridge, raw_state=iam_state)
print(f"{len(actions)} action(s):\n")
for i, a in enumerate(actions, 1):
    print(f"  [{i}] {a.action_type.upper()}: {a.from_state} → {a.to_state}")
    print(f"       {a.reason[:95]}")
    print(f"       {a.uag_link[:80]}\n")

In [ ]:
deltas = propose_delta([a.to_dict() for a in actions])
print("Constitutional OS deltas:")
for d in deltas:
    status = "ALLOWED" if d['membrane_allowed'] else "BLOCKED"
    print(f"  [{status}] {d['id'][:8]}  {d['action_type']:20s} → {d['to_state']}")
    if not d['membrane_allowed']:
        print(f"           {d['membrane_reason']}")

## 5. Hardening Simulation

Simulate: admin_users 5→2, mfa_disabled 12→0, inactive_keys 8→0.
Then recompute curvature and compare.

In [ ]:
hardened = dict(iam_state)
hardened.update({'admin_users': 2, 'mfa_disabled_users': 0, 'inactive_access_keys': 0})
x2      = encode_state(hardened, target_dim=16)
result2 = curvature_info(x2, potential, backend='numpy')
ridge2  = analyse_ridge(result2)

print(f"{'Metric':28s}  {'Before':>10s}  {'After':>10s}  {'Δ':>10s}")
print("-" * 64)
for label, b, a in [
    ("Risk R(x)",         result.risk,              result2.risk),
    ("λ_min (softest)",   ridge.softest_eigenvalue, ridge2.softest_eigenvalue),
    ("Kramers proxy",     ridge.kramers_proxy,      ridge2.kramers_proxy),
]:
    d = a - b
    print(f"  {label:26s}  {b:10.4f}  {a:10.4f}  {d:+10.4f}")
print(f"\n  Severity: {ridge.severity} → {ridge2.severity}")

## 6. Visualisations

In [ ]:
# Before / after eigenvalue comparison
fig = before_after(result.eigenvalues[:8], result2.eigenvalues[:8])
plt.tight_layout(); plt.show()

In [ ]:
# Incremental risk timeline
risks = [result.risk]
s = dict(iam_state)
for k, cut in [('admin_users',1),('mfa_disabled_users',6),
               ('inactive_access_keys',4),('mfa_disabled_users',6)]:
    s = dict(s); s[k] = max(0, s[k] - cut)
    xi = encode_state(s, target_dim=16)
    risks.append(curvature_info(xi, potential, backend='numpy').risk)
fig = risk_over_time(risks, [f"t{i}" for i in range(len(risks))],
                     title="Risk R(x) during incremental hardening")
plt.tight_layout(); plt.show()
print("Risk trajectory:", " → ".join(f"{r:.0f}" for r in risks))

In [ ]:
# 2D risk basin (x₀, x₁ slice)
fig = curvature_basin_2d(potential, x_range=(-5,5), y_range=(-5,5), resolution=80,
                         title="Risk basin R(x) — privilege potential (x₀, x₁ slice)")
plt.tight_layout(); plt.show()

In [ ]:
# blast_radius potential — shows rotated, asymmetric basin
br = BlastRadiusRisk()
xb = encode_state(iam_state, target_dim=16)
rb = curvature_info(xb, br, backend='numpy')
fig = eigenvalue_spectrum(rb.eigenvalues[:8], title="blast_radius eigenvalues (asymmetric basin)")
plt.tight_layout(); plt.show()
print(f"Min |λ|: {min(abs(rb.eigenvalues[:8])):.4f}  → severity: {analyse_ridge(rb).severity}")

## 7. Summary

| | Before | After |
|---|---|---|
| Risk R(x) | see output | lower |
| Softest λ | see output | unchanged (quadratic potential) |
| Severity | see output | see output |

For the `blast_radius` potential with off-diagonal coupling, some λ < 1.0 —
triggering severity `watch` and privilege-reduction recommendations.
This is the potential to use when modelling network lateral movement risk.

### Next steps

```python
# Exact Hessians (100–1087× speedup):
result = curvature_info(x, potential, backend='jax-xla')

# Kubernetes hardening:
from care.adapters.k8s import encode_k8s_state
x = encode_k8s_state(fetch_k8s_state())

# Apply changes via Constitutional OS:
# CARE_COS_ENABLED=true  CARE_COS_ENDPOINT=http://cos-runtime:9000
```

**References:**
- UAG: https://doi.org/10.5281/zenodo.19394700
- hcderiv v0.4.0: https://doi.org/10.5281/zenodo.19433812